In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"


In [2]:
# Add prompt based test cases map for classification evaluation. We refund_request, order_status, billing_issue

# Add classify intent that would evaluate with AI the user message for his intention with Temperature=0
# In the end add to our existing logic prompt evaluation for the user intent so we can evaluate our flow

In [3]:
# Test cases for intent classification. Each entry pairs a realistic user
# message with the intent we expect the classifier to pick.
INTENTS = ["refund_request", "order_status", "billing_issue"]

test_cases = [
    {"message": "I want to return my order and get my money back.", "expected_intent": "refund_request"},
    {"message": "Can I get a refund for order #48213? It arrived broken.", "expected_intent": "refund_request"},
    {"message": "I'd like to send this back and be reimbursed.", "expected_intent": "refund_request"},
    {"message": "Where is my package? It hasn't arrived yet.", "expected_intent": "order_status"},
    {"message": "Can you tell me the status of order #77120?", "expected_intent": "order_status"},
    {"message": "Has my order shipped yet?", "expected_intent": "order_status"},
    {"message": "I was charged twice for the same order.", "expected_intent": "billing_issue"},
    {"message": "The amount on my credit card statement doesn't match my receipt.", "expected_intent": "billing_issue"},
    {"message": "Why was I billed $20 more than the price I saw at checkout?", "expected_intent": "billing_issue"},
]

In [4]:
CLASSIFIER_SYSTEM_PROMPT = f"""You classify a customer's message into exactly one intent.
Valid intents: {", ".join(INTENTS)}.
Respond with only the intent label and nothing else."""


def classify_intent(message: str) -> str:
    """Ask Claude to label a single user message with one of INTENTS, deterministically."""
    response = client.messages.create(
        model=model,
        max_tokens=20,
        temperature=0,
        system=CLASSIFIER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": message}],
    )

    text = next(block.text for block in response.content if block.type == "text")
    return text.strip()

In [5]:
def run_intent_eval(cases: list) -> None:
    """Run classify_intent over each test case and report pass/fail plus accuracy."""
    passed = 0

    for case in cases:
        predicted = classify_intent(case["message"])
        is_correct = predicted == case["expected_intent"]
        passed += is_correct

        status = "PASS" if is_correct else "FAIL"
        print(f"[{status}] \"{case['message']}\" -> predicted={predicted!r}, expected={case['expected_intent']!r}")

    print(f"\n{passed}/{len(cases)} passed ({passed / len(cases):.0%})")


run_intent_eval(test_cases)

[PASS] "I want to return my order and get my money back." -> predicted='refund_request', expected='refund_request'
[PASS] "Can I get a refund for order #48213? It arrived broken." -> predicted='refund_request', expected='refund_request'
[PASS] "I'd like to send this back and be reimbursed." -> predicted='refund_request', expected='refund_request'
[PASS] "Where is my package? It hasn't arrived yet." -> predicted='order_status', expected='order_status'
[PASS] "Can you tell me the status of order #77120?" -> predicted='order_status', expected='order_status'
[PASS] "Has my order shipped yet?" -> predicted='order_status', expected='order_status'
[PASS] "I was charged twice for the same order." -> predicted='billing_issue', expected='billing_issue'
[PASS] "The amount on my credit card statement doesn't match my receipt." -> predicted='billing_issue', expected='billing_issue'
[PASS] "Why was I billed $20 more than the price I saw at checkout?" -> predicted='billing_issue', expected='billing_i

In [6]:
def add_user_message(messages: list, text: str) -> None:
    """Append a user turn to the conversation."""
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages: list, text: str) -> None:
    """Append an assistant turn to the conversation."""
    messages.append({"role": "assistant", "content": text})

In [7]:
SYSTEM_PROMPT = """You are a helpful shop assistant for an online store.
When a customer asks for an order return or refund, you must first ask for
their order number before you can proceed, unless they've already given it.
Once you have the order number, confirm the return and explain the next steps."""

# The whole conversation lives in this list — we resend it on every call.
messages = []

def send_message(messages: list) -> str:
    """Call Claude with the current conversation and return the reply text."""
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=messages,
    )

    return next(block.text for block in response.content if block.type == "text")

In [8]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    # VS Code's input box returns "" for both Escape and a blank Enter, so
    # blank input doubles as the way to quit here.
    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    add_user_message(messages, user_input)
    reply = send_message(messages)
    add_assistant_message(messages, reply)

    print(f"Assistant: {reply}")

User: 
